# 06 — Export the paper bundle

Collects everything the manuscript is built from — all results CSVs, `headline.json`, the five figures, the protocol amendments, `make_diagrams.py`, and the frozen `src/` — into one zip with a SHA-256 manifest, and downloads it.

The manifest is the audit trail: any number in the paper traces to a file in this bundle, and the bundle's hashes pin exactly which version of each file produced it. This zip is also what gets archived (GitHub/Zenodo) at submission time.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, hashlib, json, zipfile, shutil
from datetime import date
import pandas as pd

ROOT = '/content/drive/MyDrive/research/ids-label-correction'
os.chdir(ROOT)
print('project root:', ROOT)

Mounted at /content/drive
project root: /content/drive/MyDrive/research/ids-label-correction


In [2]:
# --- Build MANIFEST.csv: SHA-256 + size for every file going into the paper --
def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

INCLUDE_DIRS = ['results', 'figures', 'src']
INCLUDE_FILES = ['PROTOCOL.md', 'PROTOCOL_AMENDMENTS.md', 'make_diagrams.py']

rows = []
for d in INCLUDE_DIRS:
    if not os.path.isdir(d):
        print('missing directory (skipped):', d)
        continue
    for fn in sorted(os.listdir(d)):
        p = os.path.join(d, fn)
        if os.path.isfile(p):
            rows.append({'path': p, 'bytes': os.path.getsize(p), 'sha256': sha256(p)})
for f in INCLUDE_FILES:
    if os.path.isfile(f):
        rows.append({'path': f, 'bytes': os.path.getsize(f), 'sha256': sha256(f)})
    else:
        print('missing file (skipped):', f)

man = pd.DataFrame(rows)
man.to_csv('MANIFEST.csv', index=False)
print(f'\nMANIFEST.csv: {len(man)} files, {man["bytes"].sum()/1e6:.1f} MB total')
man

missing file (skipped): PROTOCOL.md

MANIFEST.csv: 33 files, 0.9 MB total


,path,bytes,sha256
0,results/appendix_degenerate_runs.csv,485,cd265663bdf88822e6311d855258c6f3aacb411fc74d1b...
1,results/binary_headline.csv,1118,0ea5b70447263726b518df407cfd69ed1fde9a6a5b5a06...
2,results/checksums.csv,3217,01e02738379a78e1ab4b5b7a25ea6b0fc5317dee3cd752...
3,results/environment.json,131,6a4a791fc854f756801396d00db9f3d12761f819c46b4e...
4,results/gate_g1.json,131,f3c246c3088e76952d377aaaf11c63e4bf0c76f016c37b...
5,results/h1_dataset_level.csv,868,d97256d22588a7cea17d54c5d92641717223bf867080fe...
6,results/h2_label_isolation.csv,1044,39834d2d55e98c7ac120a9663651916644f4d6a737b99a...
7,results/h3_attempted_sensitivity.csv,625,e0aeb10f85567a80f6531d349bc5e09c3abde788326fa7...
8,results/h4_control_classes.json,26,bb88540fbd3c5dc2b947ea29ca3873da1e7a85f6f9c3f2...
9,results/h4_equivalence.csv,511,3a9a154c894f14ce24565623096f65ea6eacc416308bdc...


In [3]:
# --- Sanity: the exhibits the manuscript depends on must all be present ------
REQUIRED = [
    'results/runs.csv',
    'results/headline.json',
    'results/h1_dataset_level.csv',
    'results/h2_label_isolation.csv',
    'results/h3_attempted_sensitivity.csv',
    'results/h4_equivalence.csv',
    'results/binary_headline.csv',
    'results/matched_representativeness.csv',
    'results/appendix_degenerate_runs.csv',
    'results/table1_class_census.csv',
    'results/table5_label_transition_matrix.csv',
    'results/table6_match_summary.csv',
    'results/table7_relabel_rate_per_class.csv',
    'results/table0_file_reconciliation.csv',
    'results/match_calibration.json',
    'results/gate_g1.json',
    'results/repair_log.json',
    'results/environment.json',
    'results/checksums.csv',
    'figures/fig0_pipeline.png',
    'figures/fig0b_design.png',
    'figures/fig1_arm_a_vs_b.png',
    'figures/fig2_unseen_family_recall.png',
    'figures/fig3_transition_heatmap.png',
    'PROTOCOL_AMENDMENTS.md',
]

missing = [p for p in REQUIRED if not os.path.exists(p)]
for p in REQUIRED:
    print(('OK   ' if os.path.exists(p) else 'MISS ') + p)
if missing:
    print(f'\n{len(missing)} required file(s) missing — re-run the notebook that '
          'produces them before exporting.')
else:
    print('\nAll required exhibits present.')

OK   results/runs.csv
OK   results/headline.json
OK   results/h1_dataset_level.csv
OK   results/h2_label_isolation.csv
OK   results/h3_attempted_sensitivity.csv
OK   results/h4_equivalence.csv
OK   results/binary_headline.csv
OK   results/matched_representativeness.csv
OK   results/appendix_degenerate_runs.csv
OK   results/table1_class_census.csv
OK   results/table5_label_transition_matrix.csv
OK   results/table6_match_summary.csv
OK   results/table7_relabel_rate_per_class.csv
OK   results/table0_file_reconciliation.csv
OK   results/match_calibration.json
OK   results/gate_g1.json
OK   results/repair_log.json
OK   results/environment.json
OK   results/checksums.csv
OK   figures/fig0_pipeline.png
OK   figures/fig0b_design.png
OK   figures/fig1_arm_a_vs_b.png
OK   figures/fig2_unseen_family_recall.png
OK   figures/fig3_transition_heatmap.png
OK   PROTOCOL_AMENDMENTS.md

All required exhibits present.


In [4]:
# --- Zip and download --------------------------------------------------------
stamp = date.today().strftime('%Y%m%d')
zpath = f'/content/paper_bundle_{stamp}.zip'

with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for d in INCLUDE_DIRS:
        if not os.path.isdir(d):
            continue
        for fn in sorted(os.listdir(d)):
            p = os.path.join(d, fn)
            if os.path.isfile(p):
                z.write(p)
    for f in INCLUDE_FILES + ['MANIFEST.csv']:
        if os.path.isfile(f):
            z.write(f)

size = os.path.getsize(zpath) / 1e6
print(f'{zpath}: {size:.1f} MB')
with zipfile.ZipFile(zpath) as z:
    print(len(z.namelist()), 'files inside')

from google.colab import files
files.download(zpath)

/content/paper_bundle_20260822.zip: 0.7 MB
34 files inside


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>